# StageOps: limpieza de `system_changes.csv`

**Hackatón SO-2026-041 · Investigación de incidencias en la venta de entradas**

## Contexto
`system_changes.csv` es el historial de **cambios de configuración** de las compañías de ticketing: una fila por cambio, con su fecha de entrada en vigor, el componente afectado (web, pagos, emisión de entradas), la versión y el valor anterior y nuevo del parámetro. Cada cambio afecta a **todos los conciertos de la compañía**.

Es la fuente que permite el apartado 03 del encargo: *evaluar el historial de cambios sin atribuir causalidad solo por una coincidencia de fechas*. Aquí no se busca la causa de nada: se deja el historial **verificado y convertido en intervalos de vigencia** para que la integración pueda saber qué configuración estaba activa en cada instante.

`CSV original → Inspección → Tipado → Validación → Intervalos de vigencia → Consulta "configuración vigente" → Tabla limpia`

El CSV original (`data/raw/`) no se modifica.

## 1. Método de trabajo

1. **Inspeccionar** el fichero, que al tener seis filas se puede leer entero.
2. **Tipar** fechas (UTC) y valores; el texto vacío de `old_value` significa «estado inicial documentado».
3. **Validar** formato y, sobre todo, la **cadena de cambios**: el `old_value` de cada cambio debe coincidir con el `new_value` del anterior del mismo parámetro.
4. **Construir intervalos** `[valid_from, valid_to)` por compañía, componente y parámetro.
5. **Definir la consulta** «configuración vigente en el instante t» y probarla en los bordes del intervalo.
6. **Validar con `assert`** y **exportar**.

El anexo técnico avisa de que unir por compañía a secas multiplicaría cada operación por todos los cambios históricos: la configuración se determina por **compañía, componente, parámetro y fecha**, tomando el último cambio efectivo anterior o igual al instante analizado.

## 2. Importaciones y ubicación de los archivos

La raíz del repositorio se localiza subiendo desde la carpeta de trabajo hasta encontrar `data/raw`. Las rutas se definen una sola vez como constantes.

In [ ]:
from pathlib import Path

import pandas as pd

CARPETA_BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'raw').is_dir())
RUTA_ORIGINAL = CARPETA_BASE / 'data' / 'raw' / 'system_changes.csv'
RUTA_CONCIERTOS = CARPETA_BASE / 'data' / 'raw' / 'concerts.csv'
CARPETA_LIMPIOS = CARPETA_BASE / 'data' / 'clean'
RUTA_LIMPIO = CARPETA_LIMPIOS / 'system_changes_clean.csv'

SNAPSHOT_AT = pd.Timestamp('2026-10-28 23:59:59', tz='UTC')   # cierre de datos indicado en el encargo
FORMATO_FECHA = '%Y-%m-%dT%H:%M:%SZ'                          # ISO 8601 UTC
CLAVE_PARAMETRO = ['ticketing_company_id', 'component', 'setting_name']

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 200)

if not RUTA_ORIGINAL.exists():
    raise FileNotFoundError(f'No se encuentra {RUTA_ORIGINAL}. Revisa el directorio de trabajo: {Path.cwd()}')
CARPETA_LIMPIOS.mkdir(parents=True, exist_ok=True)
print('Carpeta base:', CARPETA_BASE)

## 3. Inspección del fichero original

Se lee con `encoding='utf-8-sig'` (el CSV lleva BOM). `old_value` y `new_value` se leen como **texto**: unas veces contienen números (`60`, `45`) y otras nombres de regla (`CURRENT_OWNER_AND_NOT_EXPIRED`), y mezclarlos en una columna numérica los corrompería. Las filas no están ordenadas cronológicamente, tal como avisa el enunciado.

In [ ]:
df_raw = pd.read_csv(RUTA_ORIGINAL, encoding='utf-8-sig', dtype={'old_value': 'string', 'new_value': 'string', 'version': 'string'})

print(f'Registros: {len(df_raw)} | columnas: {list(df_raw.columns)}\n')
df_raw.info()
display(df_raw.sort_values('effective_at'))
print('Valores ausentes por columna:', df_raw.isna().sum()[lambda s: s > 0].to_dict())

**Lo que muestran los datos:** seis cambios, tres por compañía (LT y EL), sobre tres componentes: `ticket_issuer` (emisión), `web` y `payments`. El único valor ausente es `old_value` en dos filas, y no es un defecto: el diccionario indica que queda vacío cuando se documenta el estado inicial (`CHG-001` y `CHG-002`, «Configuración inicial del periodo exportado»).

Hay un cambio que destaca por su contenido: **`CHG-004`** modifica en EntradaLink el parámetro `reservation_validation` de `CURRENT_OWNER_AND_NOT_EXPIRED` a `PAYMENT_REFERENCE_EXISTS`. Según el diccionario, la primera regla exige vigencia y propiedad actual de la localidad y la segunda solo comprueba que exista la referencia de pago. Se estudia en el apartado 7, sin sacar conclusiones de causalidad.

## 4. Tipado y normalización

Se trabaja sobre una copia. `effective_at` pasa a `datetime64` en **UTC**. Se eliminan espacios sobrantes y se ordena por compañía, componente, parámetro y fecha, que es el orden natural para reconstruir la cadena de cambios.

In [ ]:
df_changes = df_raw.copy()

for columna in ['change_id', 'ticketing_company_id', 'component', 'version', 'setting_name', 'old_value', 'new_value', 'description']:
    df_changes[columna] = df_changes[columna].astype('string').str.strip()

df_changes['effective_at'] = pd.to_datetime(df_changes['effective_at'], utc=True, errors='coerce').astype('datetime64[ns, UTC]')
df_changes = df_changes.sort_values([*CLAVE_PARAMETRO, 'effective_at']).reset_index(drop=True)

df_changes.dtypes.to_frame('tipo')

## 5. Validaciones de calidad

Además de los formatos, se comprueba la **cadena de cambios**: dentro de un mismo parámetro, el `old_value` de cada cambio debe ser el `new_value` del anterior. Si no lo fuera, faltaría algún cambio en el historial y no se podría reconstruir con fiabilidad qué configuración estaba activa en cada momento.

In [ ]:
companias_catalogo = set(pd.read_csv(RUTA_CONCIERTOS, encoding='utf-8-sig')['ticketing_company_id'])

grupo = df_changes.groupby(CLAVE_PARAMETRO)
df_changes['prev_new_value'] = grupo['new_value'].shift(1)
df_changes['es_primero'] = grupo.cumcount() == 0

comprobaciones = pd.Series({
    'change_id duplicado': df_changes['change_id'].duplicated().sum(),
    'Formato de change_id distinto de CHG-NNN': (~df_changes['change_id'].str.match(r'^CHG-\d{3}$')).sum(),
    'Fecha no interpretable': df_changes['effective_at'].isna().sum(),
    'Cambio posterior al cierre de datos': (df_changes['effective_at'] > SNAPSHOT_AT).sum(),
    'Compañía que no existe en concerts.csv': (~df_changes['ticketing_company_id'].isin(companias_catalogo)).sum(),
    'Versión sin formato X.Y.Z': (~df_changes['version'].str.match(r'^\d+\.\d+\.\d+$')).sum(),
    'new_value vacío': df_changes['new_value'].isna().sum(),
    'Cambio sin efecto (old_value == new_value)': (df_changes['old_value'] == df_changes['new_value']).sum(),
    'Dos cambios del mismo parámetro en el mismo instante': df_changes.duplicated([*CLAVE_PARAMETRO, 'effective_at']).sum(),
    'Cadena rota: old_value != new_value anterior': (~df_changes['es_primero'] & (df_changes['old_value'] != df_changes['prev_new_value'])).sum(),
}, name='filas que incumplen')

display(comprobaciones.to_frame())

print('Primer cambio de cada parámetro (estado previo):')
display(df_changes.loc[df_changes['es_primero'], [*CLAVE_PARAMETRO, 'change_id', 'effective_at', 'old_value', 'new_value']])

**Resultado de la calidad:** todas las reglas se cumplen y la cadena de cambios es coherente. Hay una **particularidad importante** al leer el primer cambio de cada parámetro:

- En `reservation_validation` (emisión de entradas), las dos compañías tienen un estado inicial **documentado** como fila (`CHG-001` y `CHG-002`, con `old_value` vacío).
- En `cache_ttl_seconds` (web) y `callback_retry_seconds` (pagos) **no hay fila de estado inicial**: el primer cambio sí trae un `old_value` (60 y 30) que indica el valor anterior, pero ese estado previo no tiene fecha de entrada en vigor.

Para el análisis no es un problema, porque los cambios de caché y de reintentos no son el foco; pero hay que saberlo para no suponer que todos los parámetros tienen un estado inicial fechado.

## 6. Intervalos de vigencia

Cada cambio mantiene su efecto hasta el siguiente cambio **del mismo componente y parámetro**. Se calcula para cada fila:

| Columna | Definición |
| :--- | :--- |
| `valid_from` | `effective_at` (incluido) |
| `valid_to` | `effective_at` del siguiente cambio del mismo parámetro (excluido); vacío si sigue vigente al cierre de datos |
| `is_initial` | el cambio documenta el estado inicial (`old_value` vacío) |
| `is_current_at_snapshot` | la configuración sigue vigente al cierre de datos |

La vigencia se interpreta como **`[inicio, fin)`**, igual que en el resto de ficheros.

In [ ]:
df_changes['valid_from'] = df_changes['effective_at']
df_changes['valid_to'] = df_changes.groupby(CLAVE_PARAMETRO)['effective_at'].shift(-1)
df_changes['is_initial'] = df_changes['old_value'].isna()
df_changes['is_current_at_snapshot'] = df_changes['valid_to'].isna()

df_cambios = (df_changes
              .drop(columns=['prev_new_value', 'es_primero'])
              .sort_values('effective_at')
              .reset_index(drop=True))

display(df_cambios[['change_id', 'ticketing_company_id', 'component', 'version', 'setting_name',
                    'old_value', 'new_value', 'valid_from', 'valid_to', 'is_initial', 'is_current_at_snapshot']])

**Lectura de los intervalos:** en `reservation_validation`, LumaTicket mantiene `CURRENT_OWNER_AND_NOT_EXPIRED` durante todo el periodo, mientras que EntradaLink lo mantiene hasta el **16 de octubre de 2026 a las 10:20 UTC** y desde ese instante pasa a `PAYMENT_REFERENCE_EXISTS`. Es el único cambio de la regla de validación de reservas en todo el historial.

Eso convierte a LumaTicket en una **referencia natural de comparación**: si hubiera una diferencia de comportamiento en EntradaLink a partir de esa fecha que no aparece en LumaTicket, tendría sentido estudiarla; si apareciera en ambas, el cambio no la explicaría. Esto es una pauta de análisis, no una conclusión.

## 7. Consulta «configuración vigente en el instante t»

Es la función que necesitará el notebook de integración. Dado un conjunto de instantes por compañía, devuelve el valor de un parámetro vigente en cada uno, con `pd.merge_asof`: toma, para cada instante, el **último cambio con `effective_at` anterior o igual**. Así no se multiplican las operaciones por el número de cambios históricos (el aviso del anexo técnico).

Antes de usarla se prueba en los **bordes del intervalo**: un segundo antes de `CHG-004` debe devolver la regla antigua, y en el segundo exacto del cambio, la nueva (inicio incluido).

In [ ]:
def configuracion_vigente(cambios, instantes, componente, parametro, col_compania='ticketing_company_id', col_instante='instante'):
    """Se devuelve, para cada fila de `instantes`, el valor de `parametro` vigente en ese instante."""
    historial = (cambios[(cambios['component'] == componente) & (cambios['setting_name'] == parametro)]
                 [['ticketing_company_id', 'effective_at', 'new_value', 'version', 'change_id']]
                 .rename(columns={'new_value': 'valor_vigente', 'version': 'version_vigente', 'change_id': 'change_id_vigente'})
                 .sort_values('effective_at'))
    consulta = instantes.copy()
    consulta['_orden'] = range(len(consulta))   # merge_asof exige ordenar por instante; así se recupera el orden original
    consulta[col_instante] = consulta[col_instante].astype('datetime64[ns, UTC]')
    consulta[col_compania] = consulta[col_compania].astype('string')   # mismo tipo que el historial, o merge_asof falla
    historial['ticketing_company_id'] = historial['ticketing_company_id'].astype('string')
    consulta = consulta.sort_values(col_instante)
    union = pd.merge_asof(consulta, historial, left_on=col_instante, right_on='effective_at',
                          left_by=col_compania, right_by='ticketing_company_id', direction='backward',
                          allow_exact_matches=True)
    return union.sort_values('_orden').drop(columns='_orden').reset_index(drop=True)


cambio_4 = df_cambios.loc[df_cambios['change_id'] == 'CHG-004', 'effective_at'].iloc[0]
sonda = pd.DataFrame({
    'ticketing_company_id': ['EL', 'EL', 'EL', 'LT', 'LT'],
    'instante': pd.to_datetime([cambio_4 - pd.Timedelta(seconds=1), cambio_4, SNAPSHOT_AT,
                                cambio_4 - pd.Timedelta(seconds=1), SNAPSHOT_AT], utc=True),
    'caso': ['EL un segundo antes de CHG-004', 'EL en el segundo exacto de CHG-004', 'EL al cierre de datos',
             'LT un segundo antes', 'LT al cierre de datos'],
})

resultado = configuracion_vigente(df_cambios, sonda, 'ticket_issuer', 'reservation_validation')
display(resultado[['caso', 'instante', 'valor_vigente', 'version_vigente', 'change_id_vigente']])

# Comprobaciones de borde, para que el notebook falle si la consulta no respeta [inicio, fin)
esperado = ['CURRENT_OWNER_AND_NOT_EXPIRED', 'PAYMENT_REFERENCE_EXISTS', 'PAYMENT_REFERENCE_EXISTS',
            'CURRENT_OWNER_AND_NOT_EXPIRED', 'CURRENT_OWNER_AND_NOT_EXPIRED']
assert resultado['valor_vigente'].tolist() == esperado, 'la consulta de configuración vigente no respeta [inicio, fin)'
assert len(resultado) == len(sonda), 'la consulta ha multiplicado o perdido filas'
print('Consulta validada: respeta el inicio incluido y no multiplica filas.')

**Resultado:** la consulta devuelve la regla antigua un segundo antes del cambio y la nueva en el segundo exacto, tal como exige `[inicio, fin)`. Para LumaTicket devuelve siempre la regla original, y para cada instante de entrada devuelve **exactamente una fila**, por lo que no multiplica operaciones.

Solo está pensada para parámetros con estado inicial documentado (`reservation_validation`). Para `cache_ttl_seconds` y `callback_retry_seconds`, un instante anterior al primer cambio devolvería vacío; el valor previo está en el `old_value` de ese primer cambio.

## 8. Validación y exportación

Antes de exportar se comprueban con `assert` las propiedades que necesitarán los demás notebooks. Se guarda `system_changes_clean.csv` con las fechas en ISO 8601 UTC y se relee para comprobar que conserva filas y columnas.

In [ ]:
def validar_cambios(df):
    """Se comprueba que el historial es usable como tabla de referencia; cada assert detiene el proceso si falla."""
    def comprobar(condicion, descripcion):
        assert condicion, f'FALLO: {descripcion}'
        print(f'  ok · {descripcion}')

    comprobar(len(df) == 6, 'hay seis cambios, como en el fichero original')
    comprobar(df['change_id'].is_unique, 'change_id es único (clave primaria)')
    comprobar(df.groupby(CLAVE_PARAMETRO)['effective_at'].apply(lambda s: s.is_unique).all(), 'no hay dos cambios del mismo parámetro en el mismo instante')
    comprobar((df['valid_to'].isna() | (df['valid_to'] > df['valid_from'])).all(), 'todo intervalo tiene fin posterior a su inicio')
    comprobar(df.groupby(CLAVE_PARAMETRO)['is_current_at_snapshot'].sum().eq(1).all(), 'cada parámetro tiene exactamente una configuración vigente al cierre')
    comprobar(df.groupby(CLAVE_PARAMETRO)['valid_from'].apply(lambda s: s.is_monotonic_increasing).all(), 'los intervalos de cada parámetro están encadenados en el tiempo')
    comprobar(df['is_initial'].sum() == 2, 'hay dos estados iniciales documentados (emisión en LT y EL)')


validar_cambios(df_cambios)

salida = df_cambios.copy()
for columna in salida.select_dtypes(include='datetimetz').columns:
    salida[columna] = salida[columna].dt.strftime(FORMATO_FECHA)
salida.to_csv(RUTA_LIMPIO, index=False, encoding='utf-8')

relectura = pd.read_csv(RUTA_LIMPIO)
assert relectura.shape == salida.shape and list(relectura.columns) == list(salida.columns), 'el fichero cambió al exportar'
print(f'\n{RUTA_LIMPIO.name}: {relectura.shape[0]} filas · {relectura.shape[1]} columnas')

## 9. Reflexión final

**Qué se ha demostrado**
- El historial tiene seis cambios, sin nulos inesperados, duplicados ni cadenas rotas, y los intervalos de vigencia quedan reconstruidos como `[valid_from, valid_to)`.
- En todo el periodo hay **un único cambio** de la regla de validación de reservas: `CHG-004`, en EntradaLink, el 2026-10-16 a las 10:20 UTC, de `CURRENT_OWNER_AND_NOT_EXPIRED` a `PAYMENT_REFERENCE_EXISTS`. LumaTicket no cambia esa regla.
- La consulta `configuracion_vigente` devuelve una fila por instante y respeta los bordes del intervalo.

**Qué no se ha demostrado**
- Que `CHG-004` cause ninguna incidencia. Este fichero solo dice **cuándo** cambió la regla y **qué** decía el diccionario sobre cada valor; no hay ni una operación en él. «Un cambio temporalmente próximo no demuestra por sí mismo la causa de una incidencia» (diccionario de datos).

**Cómo usarlo en la integración** (detalle en [docs/system_changes_conclusiones.md](../docs/system_changes_conclusiones.md))
- Unir por **compañía, componente, parámetro y tiempo** con `configuracion_vigente` (`merge_asof`), nunca solo por compañía.
- El instante a analizar depende de la pregunta: `issued_at` del ticket para saber con qué regla se emitió, o la recepción de la confirmación para la regla de caducidad.

**Pendiente para la integración** (requiere otros ficheros)
- Comparar tickets emitidos en EntradaLink antes y después de `CHG-004` frente a LumaTicket en las mismas fechas.
- Contrastar `issuer_version` de `tickets.csv` con las versiones de `ticket_issuer` de este fichero (4.7.2 y 4.8.0 en EL; 7.3.1 en LT).
- Comprobar si hay incidencias en LumaTicket sin ningún cambio de configuración: si las hubiera, el cambio no las explicaría.